# 10 — Síntese e ranking para o TCC

Consolidação dos achados e recomendação por cenário de uso.

In [1]:
# Configuração comum dos estudos integrados
from pathlib import Path
import re
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.22

ARTEFATOS = {
    'unificado': 'result_unificado_final.xlsx',
    'ferramenta': 'result_ferramenta_final.xlsx',
    'chats': 'result_chats_final.xlsx',
}

PASTAS_PROVAVEIS = [
    Path('../artefatos'),
    Path('../../artefatos'),
    Path('artefatos'),
    Path('../'),
    Path('.'),
    Path('/mnt/data'),
]

def encontra_arquivo(nome):
    for pasta in PASTAS_PROVAVEIS:
        caminho = pasta / nome
        if caminho.exists():
            return caminho.resolve()
    raise FileNotFoundError(
        f'Arquivo não encontrado: {nome}. Coloque os artefatos em ../artefatos/, artefatos/ ou na mesma pasta do notebook.'
    )

NOMES_MODELOS = {
    'claude-haiku-4-5': 'Claude Haiku',
    'claude-opus-4-7': 'Claude Opus',
    'claude-sonnet-4-6': 'Claude Sonnet',
    'deepseek-v4-flash': 'DeepSeek Flash',
    'deepseek-v4-pro': 'DeepSeek Pro',
    'gpt-4o-mini': 'GPT 4o mini',
    'gpt-5.4': 'GPT 5.4',
    'gpt-5.4-mini': 'GPT 5.4 mini',
    'gpt-5.5': 'GPT 5.5',
    'std_chatgpt': 'ChatGPT comercial',
    'std_claude': 'Claude comercial',
}

def provedor(modelo):
    m = str(modelo).lower()
    if 'claude' in m:
        return 'Anthropic'
    if 'gpt' in m or 'chatgpt' in m:
        return 'OpenAI'
    if 'deepseek' in m:
        return 'DeepSeek'
    return 'Outro'

def prepara(df, origem_padrao=None):
    """Cria campos de leitura do TCC em memória. Não salva nenhuma base intermediária."""
    df = df.copy()
    if 'origem_resultado' not in df.columns:
        df['origem_resultado'] = origem_padrao or 'nao_informada'
    for col in [
        'avaliacao_final', 'concisao_score', 'avaliacao_gpt', 'avaliacao_opus',
        'avaliacao_humana', 'resposta_tokens_tiktoken', 'input_tokens', 'output_tokens',
        'custo_estimado_usd', 'n_invocacoes', 'latencia_s'
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('nao pertinente', np.nan), errors='coerce')
    if 'convergencia' in df.columns:
        df['convergencia'] = df['convergencia'].astype(bool)
    else:
        df['convergencia'] = np.nan
    df['acerto'] = df['avaliacao_final']
    df['resposta_direta'] = df['concisao_score']
    df['discordancia_avaliadores'] = ~df['convergencia']
    df['origem_legivel'] = df['origem_resultado'].map({
        'ferramenta': 'Ferramenta',
        'chat_comercial': 'Chat comercial'
    }).fillna(df['origem_resultado'])
    df['modelo_legivel'] = df['modelo'].map(NOMES_MODELOS).fillna(df['modelo'])
    df['provedor'] = df['modelo'].map(provedor)
    df['status_acerto'] = pd.cut(
        df['acerto'], bins=[-0.01, 0.01, 0.99, 1.01],
        labels=['Incorreta', 'Parcial', 'Correta']
    )
    df['acerto_total'] = (df['acerto'] == 1).astype(int)
    df['erro_total'] = (df['acerto'] == 0).astype(int)
    df['parcial'] = (df['acerto'] == 0.5).astype(int)
    df['nao_totalmente_correta'] = (df['acerto'] < 1).astype(int)
    return df

def ler_artefatos():
    caminhos = {k: encontra_arquivo(v) for k, v in ARTEFATOS.items()}
    df_unificado = prepara(pd.read_excel(caminhos['unificado']))
    df_ferramenta = prepara(pd.read_excel(caminhos['ferramenta']), 'ferramenta')
    df_chats = prepara(pd.read_excel(caminhos['chats']), 'chat_comercial')
    return caminhos, df_unificado, df_ferramenta, df_chats

caminhos, df, df_ferramenta, df_chats = ler_artefatos()
print('Artefatos lidos:')
for nome, caminho in caminhos.items():
    print(f'- {nome}: {caminho}')
print(f'Base unificada: {df.shape[0]} linhas x {df.shape[1]} colunas')

# Ordem padrão dos modelos: maior acerto médio primeiro.
ORDEM_MODELOS = (
    df.groupby('modelo_legivel')['acerto']
      .mean()
      .sort_values(ascending=False)
      .index
      .tolist()
)

def pct(x):
    if pd.isna(x):
        return ''
    return f'{100*x:.1f}%'

def dinheiro(x):
    if pd.isna(x):
        return ''
    return f'US$ {x:.5f}'

def resumo_metricas(data, grupo):
    g = data.groupby(grupo, observed=True)
    out = g.agg(
        n=('acerto', 'size'),
        acerto_medio=('acerto', 'mean'),
        taxa_correta=('acerto_total', 'mean'),
        taxa_parcial=('parcial', 'mean'),
        taxa_incorreta=('erro_total', 'mean'),
        resposta_direta=('resposta_direta', 'mean'),
        discordancia=('discordancia_avaliadores', 'mean'),
        tokens_resposta=('resposta_tokens_tiktoken', 'mean'),
    ).reset_index()
    return out

def ordena_por_modelo(series_or_df, col=None):
    if isinstance(series_or_df, pd.Series):
        return series_or_df.reindex([m for m in ORDEM_MODELOS if m in series_or_df.index])
    return series_or_df.set_index(col).reindex([m for m in ORDEM_MODELOS if m in series_or_df[col].values]).reset_index()

def barh_series(s, titulo, xlabel='', percentual=False, figsize=(8, 5), limite=None):
    s = s.dropna().copy()
    if limite is not None:
        s = s.sort_values(ascending=False).head(limite)
    else:
        s = s.sort_values(ascending=True)
    vals = s * 100 if percentual else s
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(vals.index.astype(str), vals.values)
    ax.set_title(titulo, loc='left')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('')
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()

def bar_series(s, titulo, xlabel='', ylabel='', percentual=False, figsize=(8, 4), rot=0):
    s = s.dropna().copy()
    vals = s * 100 if percentual else s
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(vals.index.astype(str), vals.values)
    ax.set_title(titulo, loc='left')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=rot)
    ax.grid(axis='x', visible=False)
    plt.tight_layout()
    plt.show()

def stacked_percent(data, index_col, column_col, titulo, ordem_colunas=None, figsize=(9, 6)):
    tab = pd.crosstab(data[index_col], data[column_col], normalize='index')
    if ordem_colunas:
        tab = tab.reindex(columns=[c for c in ordem_colunas if c in tab.columns])
    tab = tab.loc[tab.sum(axis=1).sort_values().index]
    ax = (tab * 100).plot(kind='barh', stacked=True, figsize=figsize)
    ax.set_title(titulo, loc='left')
    ax.set_xlabel('% das respostas')
    ax.set_ylabel('')
    ax.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.grid(axis='y', visible=False)
    plt.tight_layout()
    plt.show()
    return tab

def heatmap_tabela(pivot, titulo, formato='.1f', percentual=True, figsize=(9, 5)):
    dados = pivot.copy()
    valores = dados.values * 100 if percentual else dados.values
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(valores, aspect='auto')
    ax.set_title(titulo, loc='left')
    ax.set_xticks(np.arange(dados.shape[1]))
    ax.set_xticklabels(dados.columns.astype(str), rotation=35, ha='right')
    ax.set_yticks(np.arange(dados.shape[0]))
    ax.set_yticklabels(dados.index.astype(str))
    # Só anota quando a matriz é pequena o suficiente para não virar poluição visual.
    if dados.shape[0] * dados.shape[1] <= 80:
        for i in range(dados.shape[0]):
            for j in range(dados.shape[1]):
                val = valores[i, j]
                if not np.isnan(val):
                    texto = f'{val:{formato}}' + ('%' if percentual else '')
                    ax.text(j, i, texto, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout()
    plt.show()

def boxplot_por_grupo(data, valor, grupo, titulo, ylabel='', figsize=(9, 5), ordem=None):
    base = data[[valor, grupo]].dropna()
    if ordem is None:
        ordem = base.groupby(grupo)[valor].median().sort_values().index.tolist()
    grupos = [base.loc[base[grupo] == g, valor].values for g in ordem if g in base[grupo].unique()]
    labels = [g for g in ordem if g in base[grupo].unique()]
    fig, ax = plt.subplots(figsize=figsize)
    ax.boxplot(grupos, labels=labels, vert=False, showfliers=False)
    ax.set_title(titulo, loc='left')
    ax.set_xlabel(ylabel or valor)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

def tabela_formatada(data, percent_cols=None, money_cols=None, round_cols=None, n=20):
    out = data.copy().head(n)
    percent_cols = percent_cols or []
    money_cols = money_cols or []
    round_cols = round_cols or []
    for c in percent_cols:
        if c in out.columns:
            out[c] = out[c].map(pct)
    for c in money_cols:
        if c in out.columns:
            out[c] = out[c].map(dinheiro)
    for c in round_cols:
        if c in out.columns:
            out[c] = out[c].round(3)
    display(out)

def texto_curto(s, n=180):
    if pd.isna(s):
        return ''
    s = re.sub(r'\s+', ' ', str(s)).strip()
    return s if len(s) <= n else s[:n-1] + '…'

Artefatos lidos:
- unificado: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_unificado_final.xlsx
- ferramenta: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_ferramenta_final.xlsx
- chats: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/eval/artefatos/result_chats_final.xlsx
Base unificada: 1650 linhas x 34 colunas


## Pergunta do estudo
Considerando todas as dimensões, quais modelos são mais recomendáveis e em quais cenários?

In [2]:
geral = df.groupby('modelo_legivel').agg(
    origem=('origem_legivel', lambda x: ', '.join(sorted(x.unique()))),
    provedor=('provedor','first'),
    n=('acerto','size'),
    acerto=('acerto','mean'),
    correta=('acerto_total','mean'),
    direta=('resposta_direta','mean'),
    discordancia=('discordancia_avaliadores','mean'),
    tokens=('resposta_tokens_tiktoken','mean')
)
# Junta métricas operacionais quando existirem.
ops = df_ferramenta.groupby('modelo_legivel').agg(
    custo=('custo_estimado_usd','mean'),
    tempo_tipico=('latencia_s','median'),
    tempo_alto=('latencia_s', lambda x: x.quantile(0.95)),
    chamadas=('n_invocacoes','mean')
)
geral = geral.join(ops, how='left')
display(geral.sort_values('acerto', ascending=False).reset_index().round(4))

,modelo_legivel,origem,provedor,n,acerto,correta,direta,discordancia,tokens,custo,tempo_tipico,tempo_alto,chamadas
0,GPT 5.5,Ferramenta,OpenAI,150,0.9967,0.9933,0.9733,0.0333,117.2067,0.0547,19.505,43.0105,2.3467
1,Claude Opus,Ferramenta,Anthropic,150,0.9767,0.9533,0.4867,0.0533,271.7067,0.1070,10.620,23.9000,2.2400
2,Claude Sonnet,Ferramenta,Anthropic,150,0.9733,0.9467,0.3600,0.0533,321.4000,0.0538,11.675,21.0900,2.3333
3,DeepSeek Flash,Ferramenta,DeepSeek,150,0.9700,0.9400,0.5600,0.0533,264.5733,0.0020,7.505,17.6455,2.4067
4,DeepSeek Pro,Ferramenta,DeepSeek,150,0.9633,0.9267,0.4733,0.0867,296.4533,0.0250,15.850,34.0580,2.4067
5,GPT 5.4,Ferramenta,OpenAI,150,0.9633,0.9400,0.9733,0.0133,107.4133,0.0236,4.890,10.6140,2.3267
6,Claude comercial,Chat comercial,Anthropic,150,0.9500,0.9000,0.0867,0.1200,581.4133,NaN,NaN,NaN,NaN
7,Claude Haiku,Ferramenta,Anthropic,150,0.9367,0.8800,0.5333,0.1133,231.8600,0.0172,12.955,60.5280,2.3067
8,GPT 4o mini,Ferramenta,OpenAI,150,0.9100,0.8467,0.8867,0.1000,140.7200,0.0014,5.200,9.7345,2.2467
9,GPT 5.4 mini,Ferramenta,OpenAI,150,0.9067,0.8400,0.9933,0.1067,100.1333,0.0060,2.665,5.1365,2.3533


In [2]:
barh_series(geral['acerto'].sort_values(), 'Ranking geral por acerto médio', 'nota média: 0 a 1', figsize=(8, 6))
barh_series(geral['direta'].sort_values(), 'Ranking geral por respostas diretas', '% de respostas diretas', percentual=True, figsize=(8, 6))

NameError: name 'geral' is not defined

In [3]:
tool_rank = geral.dropna(subset=['custo','tempo_tipico']).copy()
barh_series(tool_rank['custo'].sort_values(), 'Ranking da ferramenta por menor custo', 'US$ por resposta', figsize=(8, 6))
barh_series(tool_rank['tempo_tipico'].sort_values(), 'Ranking da ferramenta por menor tempo típico', 'segundos', figsize=(8, 6))

NameError: name 'geral' is not defined

In [4]:
# Score final para a ferramenta: equilíbrio entre qualidade, foco e operação.
eq = tool_rank.copy()
for col in ['custo','tempo_tipico','chamadas','discordancia']:
    mn, mx = eq[col].min(), eq[col].max()
    eq[col + '_norm'] = 1 - (eq[col] - mn) / (mx - mn) if mx > mn else 1
eq['score_tcc'] = (
    0.45 * eq['acerto'] +
    0.20 * eq['direta'] +
    0.10 * eq['discordancia_norm'] +
    0.10 * eq['custo_norm'] +
    0.10 * eq['tempo_tipico_norm'] +
    0.05 * eq['chamadas_norm']
)
eq = eq.sort_values('score_tcc', ascending=False)
barh_series(eq['score_tcc'].sort_values(), 'Ranking final de equilíbrio da ferramenta', 'score de 0 a 1', figsize=(8, 6))
tabela_formatada(eq.reset_index(), percent_cols=['score_tcc','acerto','correta','direta','discordancia'], money_cols=['custo'], round_cols=['tempo_tipico','tempo_alto','chamadas','tokens'], n=20)

NameError: name 'tool_rank' is not defined

In [6]:
cenarios = pd.DataFrame([
    ['Maior acerto geral', geral['acerto'].idxmax(), geral['acerto'].max()],
    ['Mais respostas diretas', geral['direta'].idxmax(), geral['direta'].max()],
    ['Menor discordância', geral['discordancia'].idxmin(), geral['discordancia'].min()],
    ['Menor custo na ferramenta', tool_rank['custo'].idxmin(), tool_rank['custo'].min()],
    ['Menor tempo típico na ferramenta', tool_rank['tempo_tipico'].idxmin(), tool_rank['tempo_tipico'].min()],
    ['Melhor equilíbrio na ferramenta', eq['score_tcc'].idxmax(), eq['score_tcc'].max()],
], columns=['Cenário de decisão', 'Modelo indicado', 'Valor'])
display(cenarios)

,Cenário de decisão,Modelo indicado,Valor
0,Maior acerto geral,GPT 5.5,0.996667
1,Mais respostas diretas,ChatGPT comercial,1.000000
2,Menor discordância,GPT 5.4,0.013333
3,Menor custo na ferramenta,GPT 4o mini,0.001448
4,Menor tempo típico na ferramenta,GPT 5.4 mini,2.665000
5,Melhor equilíbrio na ferramenta,GPT 5.4,0.917963


In [7]:
# Matriz de decisão simples para texto do TCC.
decisao = geral.copy()
decisao['qualidade'] = pd.qcut(decisao['acerto'].rank(method='first'), 3, labels=['menor acerto','acerto intermediário','maior acerto'])
decisao['foco'] = pd.qcut(decisao['direta'].rank(method='first'), 3, labels=['menos direto','direto intermediário','mais direto'])
display(decisao.reset_index()[['modelo_legivel','origem','provedor','qualidade','foco','acerto','direta','discordancia','custo','tempo_tipico']].sort_values(['qualidade','foco'], ascending=False))

,modelo_legivel,origem,provedor,qualidade,foco,acerto,direta,discordancia,custo,tempo_tipico
10,GPT 5.5,Ferramenta,OpenAI,maior acerto,mais direto,0.996667,0.973333,0.033333,0.054732,19.505
5,DeepSeek Flash,Ferramenta,DeepSeek,maior acerto,direto intermediário,0.970000,0.560000,0.053333,0.001981,7.505
2,Claude Opus,Ferramenta,Anthropic,maior acerto,menos direto,0.976667,0.486667,0.053333,0.106961,10.620
3,Claude Sonnet,Ferramenta,Anthropic,maior acerto,menos direto,0.973333,0.360000,0.053333,0.053790,11.675
8,GPT 5.4,Ferramenta,OpenAI,acerto intermediário,mais direto,0.963333,0.973333,0.013333,0.023597,4.890
4,Claude comercial,Chat comercial,Anthropic,acerto intermediário,menos direto,0.950000,0.086667,0.120000,NaN,NaN
6,DeepSeek Pro,Ferramenta,DeepSeek,acerto intermediário,menos direto,0.963333,0.473333,0.086667,0.025010,15.850
0,ChatGPT comercial,Chat comercial,OpenAI,menor acerto,mais direto,0.876667,1.000000,0.073333,NaN,NaN
9,GPT 5.4 mini,Ferramenta,OpenAI,menor acerto,mais direto,0.906667,0.993333,0.106667,0.006027,2.665
1,Claude Haiku,Ferramenta,Anthropic,menor acerto,direto intermediário,0.936667,0.533333,0.113333,0.017230,12.955


In [8]:
display(Markdown('### Frases prontas para o capítulo de resultados'))
top_acerto = geral['acerto'].idxmax()
top_direta = geral['direta'].idxmax()
top_eq = eq['score_tcc'].idxmax()
texto = f"""
- O modelo com maior acerto médio na base unificada foi **{top_acerto}**, considerando a nota final de 0 a 1.
- O modelo com maior proporção de respostas diretas foi **{top_direta}**, o que indica maior foco na pergunta.
- Entre os modelos avaliados via ferramenta, o melhor equilíbrio entre acerto, foco, custo, tempo e número de chamadas foi **{top_eq}**, segundo o score composto definido neste notebook.
- A escolha do modelo não deve se basear apenas no acerto médio: custo, tempo de resposta e dispersão da resposta alteram a recomendação conforme o cenário de uso.
"""
display(Markdown(texto))

### Frases prontas para o capítulo de resultados


- O modelo com maior acerto médio na base unificada foi **GPT 5.5**, considerando a nota final de 0 a 1.
- O modelo com maior proporção de respostas diretas foi **ChatGPT comercial**, o que indica maior foco na pergunta.
- Entre os modelos avaliados via ferramenta, o melhor equilíbrio entre acerto, foco, custo, tempo e número de chamadas foi **GPT 5.4**, segundo o score composto definido neste notebook.
- A escolha do modelo não deve se basear apenas no acerto médio: custo, tempo de resposta e dispersão da resposta alteram a recomendação conforme o cenário de uso.
